# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Get metadata as Python dict
metadata = dataset.metadata.to_json()

print(f"Dataset Title: {metadata['name']}")
print(f"Description: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values.

In [ ]:
# Enumerate all record sets in the metadata, showing their @id and associated fields
def get_record_sets(meta):
    # Croissant allows multiple ways to define record sets ('recordSet' or 'hasPart' etc)
    if 'recordSet' in meta:
        rsets = meta['recordSet']
    elif 'hasPart' in meta:
        # Sometimes record sets are under hasPart
        rsets = [x for x in meta['hasPart'] if x.get('@type') == 'RecordSet']
    else:
        rsets = []
    return rsets

def list_record_sets(meta):
    rsets = get_record_sets(meta)
    if len(rsets) == 0:
        print('No record sets found in schema. Please check the metadata structure.')
        return []
    print("Available Record Sets:")
    rs_ids = []
    for rset in rsets:
        rs_id = rset['@id']
        rs_ids.append(rs_id)
        rs_name = rset.get('name', rs_id)
        print(f" - @id: {rs_id}, name: {rs_name}")
        fields = rset.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            fname = field.get('@id', str(field))
            print(f"    Field @id: {fname}")
    return rs_ids

# List all record sets and their fields
record_sets = get_record_sets(metadata)
if not record_sets:
    # Try to reload ds w/ Croissant's API
    print("No record sets found. Attempting to inspect dynamically from loader.")
    try:
        # Croissant usually exposes them as dataset.metadata.to_json()['recordSet']
        from pprint import pprint
        pprint(metadata)
    except Exception as e:
        print(e)
else:
    rs_ids = []
    print("Record sets found:")
    for rset in record_sets:
        rs_id = rset['@id']
        rs_ids.append(rs_id)
        print(f"- {rs_id} (name: {rset.get('name', 'N/A')})")
        if 'field' in rset:
            fields = rset['field']
            if isinstance(fields, dict):
                fields = [fields]
            print("  Fields:")
            for f in fields:
                if isinstance(f, dict):
                    print(f"    - @id: {f.get('@id', '')}, name: {f.get('name', '')}")
                else:
                    print(f"    - {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# In this dataset, there may be one or more record sets. We'll attempt to load all record sets found into pandas DataFrames.

# If no record sets are defined by metadata, try to guess the main data set.

dataframes = {}
if not record_sets:
    # Most Croissant datasets have at least one RecordSet, but in rare cases
    # all data may be in a DataFile or main associated resource.
    print('No "recordSet" detected in metadata; trying to infer via dataset.record_sets if available...')
    try:
        # Try to get the first record set
        from mlcroissant._dataset import Dataset as CroissantDataset
        record_set_ids = dataset.record_sets
        print('Record sets detected via mlcroissant:', record_set_ids)
        for rs_id in record_set_ids:
            records = list(dataset.records(record_set=rs_id))
            dataframes[rs_id] = pd.DataFrame(records)
            print(f'Loaded DataFrame for record set @{rs_id} with {len(records)} records.')
    except Exception as e:
        print('Could not extract record sets:', e)
else:
    rs_ids = [rset['@id'] for rset in record_sets]
    for rs_id in rs_ids:
        try:
            records = list(dataset.records(record_set=rs_id))
            dataframes[rs_id] = pd.DataFrame(records)
            print(f'Loaded DataFrame for record set @{rs_id} with {len(records)} records.')
        except Exception as e:
            print(f'Could not load records for record set {rs_id}:', e)

# Display the columns and first few records for one of the record sets
if dataframes:
    demo_rs_id = list(dataframes.keys())[0]
    print(f'Columns for record set @{demo_rs_id}:')
    print(dataframes[demo_rs_id].columns.tolist())
    dataframes[demo_rs_id].head()
else:
    print('No DataFrames loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, normalizing distributions, and grouping data by key attributes for further analysis.

In [ ]:
# EDA: Select a numeric field for analysis. We'll attempt to guess a suitable numeric field.

import numpy as np

if dataframes:
    df = dataframes[demo_rs_id]
    # Try to automatically pick a numeric column (float/int)
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        # Try to look for numeric-looking columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f'Using numeric field for demonstration: {numeric_field}')
        threshold = df[numeric_field].mean()  # Use mean as example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a suitable categorical field if present
        group_fields = df.select_dtypes(include=[object]).columns.tolist()
        group_field = None
        for g in group_fields:
            if df[g].nunique() < 20:
                group_field = g
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean of {numeric_field} by '{group_field}':")
            print(grouped_df)
    else:
        print('No numeric field detected in DataFrame.')
else:
    print('No DataFrame available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if dataframes and numeric_fields:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=50)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouped_df exists, plot bar chart
    if 'grouped_df' in locals() and group_field is not None:
        grouped_df.plot(kind='bar')
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.show()
else:
    print('No numeric data available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was loaded from a Croissant schema and explored using `mlcroissant`.
- Record sets, fields, and columns were inspected via their `@id` references.
- Numeric fields were analyzed and normalized, with simple filtering and aggregation shown.
- Simple visualizations highlighted the data's distribution and groupwise summaries.

Continue with deeper domain-specific analysis as required to leverage the detailed experiments and protein-level data available in the dataset.